# Beam-α bootstrap (per-band `beam_scale`)

Estimate a restoring-beam scale **α** for every OVRO-LWA subband so Total fluxes
are photometrically consistent, then feed the mapping into a re-run of
`ovro_lwa_healpix_tile_detect.ipynb`.

**Method**

1. **Anchor at 73 MHz** against unique VLSSR Peak (74 MHz):
   `α_73 = BEAM_SCALE_USED × √(median Total_73 / Peak_VLSSR)`.
2. **Neighbor steps** on unconfused unique pairs (`n_assoc==1`, `n_confused==1`):
   Peak removes spectral slope via
   `(T_hi/T_lo)/(P_hi/P_lo) ≈ (α_lo/α_hi)²`.
3. Write `{CATALOG_DIR}/beam_alpha_by_band.parquet` and print a dict for paste/load.

**Prerequisites:** fusion (or spectral) metacatalog with LWA `Total_flux_*` /
`Peak_flux_*` / `n_assoc_*` / `n_confused_*`, plus unique VLSSR attach columns
(`Peak_flux_VLSSR`, `n_assoc_VLSSR`) — typically after `radio_crossmatch.ipynb`.
Set `BEAM_SCALE_USED` to the detect-time α used when that tree was built.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from lwa_catalog import CatalogLayout, read_metacatalog
from lwa_catalog.analyze import (
    CORE_CLEAN_EXCLUDE_MASK,
    BeamAlphaBootstrapConfig,
    bootstrap_beam_alpha,
    load_beam_alpha_mapping,
    summarize_beam_alpha_bootstrap,
    write_beam_alpha_table,
)

# --- operator config ---
CATALOG_DIR = Path("/fast/claw/metacatalog_healpix_IdTR-0.75s_subband2")
# Detect-time beam_scale used when this tree was built (scalar α on all bands).
BEAM_SCALE_USED = 1.2  # subband2; use 1.0 for subband3 / library default trees
ANCHOR_BAND = "73MHz"
# Prefer spectral product when VLSSR is already attached; else fusion (will fail
# without Peak_flux_VLSSR).
PREFER_SPECTRAL = True
QUALITY_FLAG_MASK = CORE_CLEAN_EXCLUDE_MASK  # None to skip quality filter on fusion
WRITE_OUTPUT = True

layout = CatalogLayout(CATALOG_DIR)
INPUT_PATH = (
    layout.metacatalog_spectral()
    if PREFER_SPECTRAL and layout.metacatalog_spectral().is_file()
    else layout.metacatalog()
)
OUTPUT_PATH = layout.beam_alpha_by_band()

print("CATALOG_DIR =", layout.root.resolve())
print("INPUT_PATH =", INPUT_PATH.resolve(), "exists:", INPUT_PATH.is_file())
print("OUTPUT_PATH =", OUTPUT_PATH.resolve())
print("BEAM_SCALE_USED =", BEAM_SCALE_USED)
print("ANCHOR_BAND =", ANCHOR_BAND)
print("PREFER_SPECTRAL =", PREFER_SPECTRAL)
print("WRITE_OUTPUT =", WRITE_OUTPUT)


## Load metacatalog + bootstrap α


In [ ]:
# Spectral product is already quality-filtered at write time.
quality_mask = None if INPUT_PATH == layout.metacatalog_spectral() else QUALITY_FLAG_MASK
meta = read_metacatalog(INPUT_PATH, quality_mask=quality_mask)
print(f"rows: {len(meta):,}  from {INPUT_PATH.name}")
print(f"Peak_flux_VLSSR present: {'Peak_flux_VLSSR' in meta.columns}")
print(f"n_assoc_VLSSR present: {'n_assoc_VLSSR' in meta.columns}")

config = BeamAlphaBootstrapConfig(
    anchor_band=ANCHOR_BAND,
    beam_scale_used=BEAM_SCALE_USED,
)
result = bootstrap_beam_alpha(meta, config=config)
print(summarize_beam_alpha_bootstrap(result))
if result.warnings:
    for w in result.warnings:
        print("warning:", w)

print("\nNeighbor steps:")
print(result.steps.to_string(index=False))
print("\nPer-band α:")
print(result.by_band.to_string(index=False))

beam_alpha_mapping = result.as_mapping()
print("\n# Paste into ovro_lwa_healpix_tile_detect.ipynb as BEAM_SCALE = {...}")
print("BEAM_SCALE = {")
for band, alpha in beam_alpha_mapping.items():
    print(f"    {band!r}: {alpha:.6g},")
print("}")


## Plot + write Parquet for re-detect


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(result.by_band["nu_mhz"], result.by_band["alpha"], "o-", color="C0", label="bootstrapped α")
ax.axhline(result.alpha_anchor, color="C1", ls="--", label=f"α_{ANCHOR_BAND}={result.alpha_anchor:.3f}")
ax.axhline(BEAM_SCALE_USED, color="C2", ls=":", label=f"BEAM_SCALE_USED={BEAM_SCALE_USED}")
ax.set_xlabel("Frequency (MHz)")
ax.set_ylabel("beam α")
ax.set_title("Per-band beam α (VLSSR anchor + neighbor Total/Peak)")
ax.legend(frameon=False)
ax.grid(True, alpha=0.3)
plt.show()

if WRITE_OUTPUT:
    out = write_beam_alpha_table(result, layout)
    print(f"Wrote {out.resolve()}")
    steps_path = out.with_name(out.stem + "_steps.parquet")
    if steps_path.is_file():
        print(f"Wrote {steps_path.resolve()}")
    # Round-trip check for detect notebook loader
    loaded = load_beam_alpha_mapping(layout)
    assert loaded == beam_alpha_mapping
    print(f"load_beam_alpha_mapping OK ({len(loaded)} bands)")
else:
    print("WRITE_OUTPUT=False — skipped parquet write")


## Re-detect checklist

1. Point `ovro_lwa_healpix_tile_detect.ipynb` `OUTPUT_DIR` at a **new** tree (or clear tile caches).
2. Set `ESTIMATE_BEAM_SCALE = False`.
3. Either:
   - `BEAM_SCALE = load_beam_alpha_mapping(CatalogLayout(OUTPUT_DIR_OF_PRIOR_TREE))`, or
   - paste the printed dict as `BEAM_SCALE = {...}`, or
   - `BEAM_SCALE_TABLE = Path(".../beam_alpha_by_band.parquet")` (see detect config).
4. Re-run tile detect → tile merge → fusion → reliability → radio crossmatch.
5. Re-run this notebook on the new tree (update `BEAM_SCALE_USED` to the mapping
   you applied, or keep the prior scalar if the table was the only α source) to
   verify neighbor Total/Peak α factors ≈ 1.
